In [ ]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformers import XLNetTokenizer
from sklearn.metrics import confusion_matrix, classification_report
import sys
import os

# Add src to path
sys.path.append(os.path.abspath('..'))

from src.dataset import get_datasets, EVASION_MAP_9, EVASION_MAP_5, CLARITY_MAP
from src.model import SingleHeadXLNet

# Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
CLARITY_NAMES = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

print(f"Using device: {DEVICE}")

In [ ]:
# 1. Load Data
tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')

# Wir laden die Datasets für alle 3 Modi
# Wichtig: Wir brauchen nur das Test-Set
_, test_ds_direct, _, _ = get_datasets('../data/processed/train.csv', '../data/processed/test.csv', tokenizer, mode='clarity')
_, test_ds_hier, _, _ = get_datasets('../data/processed/train.csv', '../data/processed/test.csv', tokenizer, mode='evasion_9')
_, test_ds_red, _, _ = get_datasets('../data/processed/train.csv', '../data/processed/test.csv', tokenizer, mode='evasion_5')

loaders = {
    "Direct": DataLoader(test_ds_direct, batch_size=BATCH_SIZE),
    "Hierarchical": DataLoader(test_ds_hier, batch_size=BATCH_SIZE),
    "Reduced": DataLoader(test_ds_red, batch_size=BATCH_SIZE)
}

# 2. Init Models
models = {
    "Direct": SingleHeadXLNet(num_labels=3).to(DEVICE),
    "Hierarchical": SingleHeadXLNet(num_labels=9).to(DEVICE),
    "Reduced": SingleHeadXLNet(num_labels=5).to(DEVICE)
}

# 3. Load Weights
# Stelle sicher, dass die Pfade stimmen!
models["Direct"].load_state_dict(torch.load('../models/xlnet_direct_clarity.pt', map_location=DEVICE))
models["Hierarchical"].load_state_dict(torch.load('../models/xlnet_hierarchical_k9.pt', map_location=DEVICE))
models["Reduced"].load_state_dict(torch.load('../models/xlnet_reduced_k5.pt', map_location=DEVICE))

print("Models loaded successfully.")

In [ ]:
# Mappings Arrays (wie im Training definiert)
MAPPING_K9 = np.array([0, 1, 1, 1, 1, 2, 2, 2, 1])
MAPPING_K5 = np.array([0, 1, 1, 1, 2])

def get_predictions(model_name):
    model = models[model_name]
    loader = loaders[model_name]
    model.eval()
    
    preds_raw = []
    trues_clarity = []
    
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            
            logits = model(ids, mask)
            preds_raw.extend(torch.argmax(logits, dim=1).cpu().numpy())
            
            # Immer die echte Clarity Truth nehmen
            trues_clarity.extend(batch['clarity_truth'].numpy())
            
    preds_raw = np.array(preds_raw)
    trues_clarity = np.array(trues_clarity)
    
    # Mapping anwenden
    if model_name == "Direct":
        preds_final = preds_raw # Kein Mapping nötig
    elif model_name == "Hierarchical":
        preds_final = MAPPING_K9[preds_raw]
    elif model_name == "Reduced":
        preds_final = MAPPING_K5[preds_raw]
        
    return trues_clarity, preds_final

# Run Predictions
results = {}
for name in models.keys():
    print(f"Predicting {name}...")
    y_true, y_pred = get_predictions(name)
    results[name] = (y_true, y_pred)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, (name, (y_true, y_pred)) in enumerate(results.items()):
    cm = confusion_matrix(y_true, y_pred)
    
    # Normalize for better visualization (optional, remove fmt='.2f' if raw numbers wanted)
    # cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=CLARITY_NAMES, yticklabels=CLARITY_NAMES)
    
    axes[i].set_title(f"{name} Approach")
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True')

plt.tight_layout()
plt.show()

In [ ]:
# Erstelle eine schöne Tabelle für den Vergleich
final_metrics = []

for name, (y_true, y_pred) in results.items():
    report = classification_report(y_true, y_pred, target_names=CLARITY_NAMES, output_dict=True)
    final_metrics.append({
        "Model": name,
        "Macro F1": report['macro avg']['f1-score'],
        "Accuracy": report['accuracy'],
        "F1 (Ambivalent)": report['Ambivalent']['f1-score'],
        "F1 (Clear Reply)": report['Clear Reply']['f1-score'],
        "F1 (Clear Non-Reply)": report['Clear Non-Reply']['f1-score']
    })

df_results = pd.DataFrame(final_metrics)
display(df_results)

# Optional: Als CSV speichern für LaTeX Tabelle
# df_results.to_csv('final_results.csv', index=False)